In [ ]:
import os   

In [ ]:
base_dir = os.getcwd()

base_dir = os.path.join(base_dir, '../')

In [ ]:

!pip install ipynb >> /dev/null
!pip install nbformat >> /dev/null
!pip install papermill >> /dev/null
!pip install rdflib >> /dev/null

In [ ]:
import papermill as pm

from rdflib import *

In [ ]:
# Set the current working directory to the notebook's directory
os.chdir(os.path.dirname(os.path.abspath('__file__')))

# DATA VALIDATION FRAMEWORK 

![title](demo_images/Prototype.png)    

The Data Validation Framework is intented to be used in federated enviorments such as federated repositories and data spaces. In the above figure, the different components, roles and workflows are represented.

Roles:

    - Data Owners (i.e, Providers)
    - Data Consumers
    - Federated Team (i.e. Federator)

Components:

    - Data Platform Layer: Acts as interface for the different roles to interact with the framework.
    - Semantic Data Model $\Delta$: Graph-based construct with all the metadata of the federation acting as a backbone to operacionalize processes.
    - Data Validation Workflow includes the following:
        - Parser: Generates Policy Checkers
        - Translator: Generates UDFs for data validation
        - Graph Transformation rules: Used to generate Policy Checkers
        - Code metadata: Used to generate UDFs

Wokflows:

    - Data federation population (Blue Arrows)
    - Data Validation Workflow (Orange Arrows)        




## GRAPH-BASED DATA FEDERATION

### Global Definitions and Policies

![title](demo_images/GD.png)

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'FederatedComputationalGovernance/federated_layer_populator.ipynb'),None, parameters={
        'folder':  os.path.join(base_dir, 'FederatedComputationalGovernance/'),
    },
    log_output=False
)

### Data product Registration

![title](demo_images/SG.png)

#### DP1

In [ ]:
file_path = r"/home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct/Data/Explotation/UPENN-GBM_clinical_info_v2.1.csv"

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Registration/profiler.ipynb'),None, parameters={
        'folder':  os.path.join(base_dir, 'DataPlatformLayer/Registration'),
        'file_path': file_path
    },
    log_output=False
)


#### DP2

In [ ]:
file_path =r"/home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct2/Data/0002.DCM"

In [ ]:

pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Registration/profiler.ipynb'), None, parameters={
        'folder': os.path.join(base_dir, 'DataPlatformLayer/Registration'),
        'file_path': file_path
    },
    log_output=False
)

![title](demo_images/SDM.png)

#### DP1 (ML Model)


In [ ]:
file_path =r"/home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct3/Data/Explotation/model.pkl"

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Registration/profiler.ipynb'), None, parameters={
        'folder': os.path.join(base_dir, 'DataPlatformLayer/Registration'),
        'file_path': file_path
    },
    log_output=False
)

In [ ]:
file_path =r"/home/acraf/psr/Fdatavalidation/DataProductLayer/DataProduct3/Data/Explotation/test.csv"

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Registration/profiler.ipynb'), None, parameters={
        'folder': os.path.join(base_dir, 'DataPlatformLayer/Registration'),
        'file_path': file_path
    },
    log_output=False
)

## Integration

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Integration/federator.ipynb'),None, parameters={
        'folder':  os.path.join(base_dir, 'DataPlatformLayer/Integration'),
        'dp_meta_path': os.path.join(base_dir, 'DataPlatformLayer/Integration/dp1.json'),
    },
    log_output=False
)

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Integration/federator.ipynb'),None, parameters={
        'folder':  os.path.join(base_dir, 'DataPlatformLayer/Integration'),
        'dp_meta_path': os.path.join(base_dir, 'DataPlatformLayer/Integration/dp2.json'),
    },
    log_output=False
)

In [ ]:
pm.execute_notebook(
    os.path.join(base_dir, 'DataPlatformLayer/Integration/federator.ipynb'),None, parameters={
        'folder':  os.path.join(base_dir, 'DataPlatformLayer/Integration'),
        'dp_meta_path': os.path.join(base_dir, 'DataPlatformLayer/Integration/dp3.json'),
    },
    log_output=False
)

## DATA VALIDATION WORKFLOW

![title](demo_images/AS.png)

In [ ]:
sdm = Graph().parse(os.path.join(base_dir, 'FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

Avaliable Common Data Models

In [ ]:
query = f'''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX ab: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#>
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>

SELECT ?dp
WHERE {{
    ?dp a tb:CommonDataModel .
}}
'''

q_res = sdm.query(query)

data_products = [row.dp.split("#")[1] for row in q_res]
data_products

The user selects a common data model for analysis

In [ ]:
#dp = input("SELECT NAME OF THE COMMON DATA MODEL: ")
dp = "Patient_Data"

We retrive the data products that are mapped to the selected common data model through their data contracts

In [ ]:
# Run the query
query = f'''
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX ab: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#>
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>

SELECT ?dataset
WHERE {{
    ?dataset tb:hasDC ?dcc .
    ?dcc tb:hasPolicy ?p .
    ?p odrl:prohibition|odrl:duty ?d .
    ?d odrl:target ?t .
    ?dp tb:hasFeature ?t .
    FILTER (?dp = ab:{dp})
}}
'''

qres = sdm.query(query)


data_products = [row.dataset.split("#")[1] for row in qres]

data_products = set(data_products)
data_products

### PARSER

The parser generates a set of policy checkers $C$ for each data product $DP$ that is mapped to the selected common data model

![title](demo_images/PC.png)

In [ ]:
for dp in data_products:
    print("DP", dp)
    pm.execute_notebook(
        os.path.join(base_dir, 'Connector/ValidationFramework/planner/planner.ipynb'),None, parameters={
            'folder': os.path.join(base_dir, 'Connector/ValidationFramework/planner/'),
            'dp': dp
        },
        log_output=False
    )   

### TRANSLATOR

The translator takes a $pc$ and generates a UDF to scrutinize $DP$

![title](demo_images/SCRIPT.png)

In [ ]:
sdm = Graph().parse(os.path.join(base_dir, 'FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

First we need to retrive all $pc \in C$ for each $DP$

In [ ]:
def queryPC(dp):
    query = f'''
    PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
    PREFIX ab: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#>
    PREFIX odrl: <http://www.w3.org/ns/odrl/2/>
    
    SELECT ?pc ?ds
    WHERE {{
        ?pc a tb:PolicyChecker .
        ?pc tb:validates ?dp .
        FILTER (?dp = ab:{dp})        
    }}
    '''

    q_res = sdm.query(query)
    
    pcs = [row.pc for row in q_res]
    return pcs
        

Execute the translator for each $pc \in C$

In [ ]:
for dp in data_products:
    pcs = queryPC(dp)
    for pc in pcs:
        print(pc.split("#")[1])
        pm.execute_notebook(             
            os.path.join(base_dir, 'Connector/ValidationFramework/translator/executor.ipynb'),None, parameters={
                'folder': '../Connector/ValidationFramework/translator/',
                'pc' : pc.split("#")[1],
            },
            log_output=False
        )

## DATA VALIDATION REPORTS

Since all the data validation metadata is stored in $\Delta$ it can be retrived through SPARQL queries

### VALIDATION RESULT

The results for each DP and its associated policies can be retrived through the following query

In [ ]:
sdm = Graph().parse(os.path.join(base_dir, 'FederatedComputationalGovernance/SemanticDataModel/sdm.ttl'), format='turtle')

In [ ]:
query = """
PREFIX odrl: <http://www.w3.org/ns/odrl/2/>
PREFIX tb: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/tbox#>
PREFIX ab: <http://www.semanticweb.org/acraf/ontologies/2024/healthmesh/abox#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?dp ?p ?pc ?result
WHERE {
    ?pc a tb:PolicyChecker .
    ?pc tb:validates ?dp .
    ?pc tb:accordingTo ?p .
    ?pc tb:nextStep+ ?rep .
    ?rep a tb:Report .
    ?rep tb:result ?result .
}
"""

res = sdm.query(query)
for row in res:
    print("DP", row["dp"].split("#")[1], "P", row["p"].split("#")[1], "Report",row["result"])

### CLARITY QUERIES

The user can query the metadata to retrive more details on how a Policy has been validated both logically and algorithmically

In [ ]:
policy = "p2"

In [ ]:
query = f"""
SELECT ?dp ?p ?pc ?op ?code
WHERE {{
     ?pc a tb:PolicyChecker . 
     ?pc tb:validates ?dp . 
     ?pc tb:accordingTo ?p . 
     FILTER (?p = ab:{policy})
     ?pc tb:nextStep+ ?op . 
     ?op tb:hasTranslation ?imp . 
     ?imp tb:hasCode ?codeclass . 
     ?codeclass tb:code ?code . 
}}
"""


res = sdm.query(query)
for row in res:
    print("DP", row["dp"].split("#")[1], "P", row["p"].split("#")[1], "OP", row["op"].split("#")[1], "Code", row["code"])